# Cleaning & EDA for Spotify Listening History Data
---
**Goal:**
- Load combined raw CSVs for both users (2021-2025)
- Clean & standardize data for analysis
- Ensure datasets are comparable and aligned in time

**Tasks:**

For both User A and User B
1. Import libraries
2. Set file paths to combined raw CSVs
3. Remove personal identifying data 
4. Inspect data:
   - Shape
   - Missing values
5. Align Time Periods
   - Limit the overlapping period (for fair comparison)
6. Data Cleaning:
    - Handle missing values
    - Timestamp to dt
    - Standardize column names
7. Save

In [83]:
#Libraries
import pandas as pd
import numpy as np
import os

## User A & B | Load
---

In [84]:
#Path to combined raw CSVs
user_a_csv = "../Combined Raw Data/User_A_Raw_2021-2025.csv"
user_b_csv = "../Combined Raw Data/User_B_Raw_2021-2025.csv"

#Load CSVs (suppress mixed-type warning)
df_user_a = pd.read_csv(user_a_csv, low_memory=False)
df_user_b = pd.read_csv(user_b_csv, low_memory=False)

print("User A loaded:", df_user_a.shape)
print("User B loaded:", df_user_b.shape)

#Remove personal identifying information
cols_to_drop = ['ip_addr', 'platform', 'conn_country']
df_user_a.drop(columns=cols_to_drop, inplace=True, errors='ignore')
df_user_b.drop(columns=cols_to_drop, inplace=True, errors='ignore')

#Check
print("User A columns after PII removal:")
print(df_user_a.columns.tolist())

print("\nUser B columns after PII removal:")
print(df_user_b.columns.tolist())

User A loaded: (143007, 23)
User B loaded: (96147, 23)
User A columns after PII removal:
['ts', 'ms_played', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'episode_name', 'episode_show_name', 'spotify_episode_uri', 'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']

User B columns after PII removal:
['ts', 'ms_played', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'episode_name', 'episode_show_name', 'spotify_episode_uri', 'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']


## Convert Timestamps & Find Overlap
---
Determined Overlap window:
**November 3, 2021 -> August 4, 2025**

Drop:
- User A data **after** August 4, 2025
- User B data **before** Nov 3, 2021

Shape Difference:
- User A: 136,517 rows
- User B: 60,383 rows

Notes:
- Minute-level differences
- Same start and end date (overlap)

In [85]:
#Convert timestamp column to dt (UTC timezone)
df_user_a['ts'] = pd.to_datetime(df_user_a['ts'], utc=True)
df_user_b['ts'] = pd.to_datetime(df_user_b['ts'], utc=True)

In [86]:
#Timestamp conversion

print("User A ts dtype:", df_user_a['ts'].dtype)
print("User B ts dtype:", df_user_b['ts'].dtype)

print("\nUser A time range:",
      df_user_a['ts'].min(), "to", df_user_a['ts'].max())

print("User B time range:",
      df_user_b['ts'].min(), "to", df_user_b['ts'].max())

#Verify any invalid timestamps
print("\nUser A invalid ts values:",
      df_user_a['ts'].isna().sum())

print("User B invalid ts values:",
      df_user_b['ts'].isna().sum())

User A ts dtype: datetime64[ns, UTC]
User B ts dtype: datetime64[ns, UTC]

User A time range: 2021-11-03 18:04:27+00:00 to 2025-09-07 15:55:28+00:00
User B time range: 2021-05-12 11:57:37+00:00 to 2025-08-04 19:59:02+00:00

User A invalid ts values: 0
User B invalid ts values: 0


In [87]:
#Ensure timestamps are dt
df_user_a['ts'] = pd.to_datetime(df_user_a['ts'], errors='coerce', utc=True)
df_user_b['ts'] = pd.to_datetime(df_user_b['ts'], errors='coerce', utc=True)

#Get time ranges (again)
a_min, a_max = df_user_a['ts'].min(), df_user_a['ts'].max()
b_min, b_max = df_user_b['ts'].min(), df_user_b['ts'].max()

print("User A time range:", a_min, "to", a_max)
print("User B time range:", b_min, "to", b_max)

#Determine shared overlap
start_date = max(a_min, b_min)
end_date = min(a_max, b_max)

print("Shared analysis window:", start_date, "to", end_date)

User A time range: 2021-11-03 18:04:27+00:00 to 2025-09-07 15:55:28+00:00
User B time range: 2021-05-12 11:57:37+00:00 to 2025-08-04 19:59:02+00:00
Shared analysis window: 2021-11-03 18:04:27+00:00 to 2025-08-04 19:59:02+00:00


In [88]:
#Filter datasets through shared window (remove all outside overlapping period)
df_user_a_aligned = df_user_a[(df_user_a['ts'] >= start_date) & (df_user_a['ts'] <= end_date)].reset_index(drop=True)
df_user_b_aligned = df_user_b[(df_user_b['ts'] >= start_date) & (df_user_b['ts'] <= end_date)].reset_index(drop=True)

#Quick check of shapes
print("User A aligned shape:", df_user_a_aligned.shape)
print("User B aligned shape:", df_user_b_aligned.shape)

User A aligned shape: (136517, 20)
User B aligned shape: (60383, 20)


In [89]:
#Aligned time range
print("User A aligned time range:", df_user_a_aligned['ts'].min(), "to", 
      df_user_a_aligned['ts'].max())
print("User B aligned time range:", df_user_b_aligned['ts'].min(), "to", 
      df_user_b_aligned['ts'].max())

User A aligned time range: 2021-11-03 18:04:27+00:00 to 2025-08-04 19:57:08+00:00
User B aligned time range: 2021-11-03 18:15:30+00:00 to 2025-08-04 19:59:02+00:00


## Cleaning
---
Tasks:

- Drop rows without activity
- Handle missing values
- Standardize column names
- Final sanity check
- Save cleaned datasets

Conclusion:
- `ts` parsed as `datetime64[ns, UTC]` for both users
- Removed 0 `ms_played`
  - User A after ms_played filter: (138724, 20)
  - User B after ms_played filter: (93367, 20)
- Missing Values
  - Both User A & B: All audiobook/podcast-related columns are ~99.9% to fully missing
  - User A: `offline_timestamp` has 5% missing
  - User B: `offline_timestamp` has 72% missing (much less offline listening data compared to user a)
  - Remove empty columns (`audiobook_title`, `audiobook_uri`, `episode_name`, etc.) & remaining missing values

In [90]:
#Remove invalid listening durations

df_user_a = df_user_a[df_user_a['ms_played'] > 0]
df_user_b = df_user_b[df_user_b['ms_played'] > 0]

print("User A after ms_played filter:", df_user_a.shape)
print("User B after ms_played filter:", df_user_b.shape)

print("User A min ms_played:", df_user_a['ms_played'].min())
print("User B min ms_played:", df_user_b['ms_played'].min())

User A after ms_played filter: (138724, 20)
User B after ms_played filter: (93367, 20)
User A min ms_played: 1
User B min ms_played: 6


In [91]:
#Inspect missing values
missing_a = df_user_a.isna().mean().sort_values(ascending=False)
missing_b = df_user_b.isna().mean().sort_values(ascending=False)

print("User A missing value % (top 10):")
print(missing_a.head(10))

print("\nUser B missing value % (top 10):")
print(missing_b.head(10))

User A missing value % (top 10):
audiobook_chapter_title              0.999986
audiobook_uri                        0.999978
audiobook_title                      0.999978
audiobook_chapter_uri                0.999978
episode_name                         0.999178
episode_show_name                    0.999178
spotify_episode_uri                  0.999178
offline_timestamp                    0.050676
master_metadata_track_name           0.000843
master_metadata_album_artist_name    0.000843
dtype: float64

User B missing value % (top 10):
audiobook_uri                        1.000000
audiobook_title                      1.000000
audiobook_chapter_uri                1.000000
audiobook_chapter_title              1.000000
episode_name                         0.999497
episode_show_name                    0.999497
spotify_episode_uri                  0.999497
offline_timestamp                    0.722729
master_metadata_track_name           0.000503
master_metadata_album_artist_name    0.00050

In [92]:
#Columns to drop because they are mostly missing
cols_to_drop_heavy_na = [
    'audiobook_title', 'audiobook_uri', 'audiobook_chapter_uri', 'audiobook_chapter_title',
    'episode_name', 'episode_show_name', 'spotify_episode_uri'
]

#Drop from both datasets
df_user_a.drop(columns=cols_to_drop_heavy_na, inplace=True, errors='ignore')
df_user_b.drop(columns=cols_to_drop_heavy_na, inplace=True, errors='ignore')

#Check
print("User A columns after dropping audiobook/podcast:", df_user_a.columns.tolist())
print("User B columns after dropping audiobook/podcast:", df_user_b.columns.tolist())

User A columns after dropping audiobook/podcast: ['ts', 'ms_played', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']
User B columns after dropping audiobook/podcast: ['ts', 'ms_played', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']


In [93]:
#Drop rows with missing track info
df_user_a.dropna(subset=['master_metadata_track_name', 'master_metadata_album_artist_name', 
    'master_metadata_album_album_name', 'spotify_track_uri'], inplace=True)
df_user_b.dropna(subset=['master_metadata_track_name', 'master_metadata_album_artist_name', 
    'master_metadata_album_album_name', 'spotify_track_uri'], inplace=True)

## Final Sanity Check
---

In [94]:
#Check
#Shapes
print("User A final shape:", df_user_a.shape)
print("User B final shape:", df_user_b.shape)

#Columns
print("User A final columns:", df_user_a.columns.tolist())
print("User B final columns:", df_user_b.columns.tolist())

#Missing values
print("\nUser A missing values (%):")
print((df_user_a.isnull().sum() / len(df_user_a) * 100).sort_values(ascending=False).head(10))

print("\nUser B missing values (%):")
print((df_user_b.isnull().sum() / len(df_user_b) * 100).sort_values(ascending=False).head(10))

#Timestamp ranges
print("\nUser A time range:", df_user_a['ts'].min(), "to", df_user_a['ts'].max())
print("User B time range:", df_user_b['ts'].min(), "to", df_user_b['ts'].max())

User A final shape: (138607, 13)
User B final shape: (93320, 13)
User A final columns: ['ts', 'ms_played', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']
User B final columns: ['ts', 'ms_played', 'master_metadata_track_name', 'master_metadata_album_artist_name', 'master_metadata_album_album_name', 'spotify_track_uri', 'reason_start', 'reason_end', 'shuffle', 'skipped', 'offline', 'offline_timestamp', 'incognito_mode']

User A missing values (%):
offline_timestamp                    5.037264
ts                                   0.000000
ms_played                            0.000000
master_metadata_track_name           0.000000
master_metadata_album_artist_name    0.000000
master_metadata_album_album_name     0.000000
spotify_track_uri                    0.000000
reason_start                         0.000000
rea

In [95]:
#Save cleaned datasets

os.makedirs("../Cleaned Data", exist_ok=True)

df_user_a_aligned.to_csv(
    "../Cleaned Data/User_A_Cleaned_Aligned_2021-2025.csv",
    index=False
)

df_user_b_aligned.to_csv(
    "../Cleaned Data/User_B_Cleaned_Aligned_2021-2025.csv",
    index=False
)

print("\nCleaned & aligned datasets saved!")


Cleaned & aligned datasets saved!
